# Installing/Importing Libraries

In [ ]:
!pip install -U sentence-transformers

In [ ]:
import os
import torch
import polars as pl
import numpy as np
from sentence_transformers import SentenceTransformer
from google.colab import drive
import time

In [ ]:
drive.mount('/content/drive')

In [ ]:
drive_path = '/content/drive/Shareddrives/CMPE256_FinalProject/board_game_recommendation'
foundation_path = f"{drive_path}/processed_data/final_foundation"
output_path = f"{drive_path}/processed_data/variant_llm_v2"
raw_data_path = f"{drive_path}/raw_data"
os.makedirs(output_path, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Loading datasets

In [ ]:
items_df = pl.read_parquet(f"{foundation_path}/splits/items_metadata_final.parquet")
items_df = items_df.with_columns(pl.col("Description").fill_null("No description available"))



## Load Subcategories

In [ ]:
subcats = pl.read_csv(f"{raw_data_path}/subcategories.csv")
designers = pl.read_csv(f"{raw_data_path}/designers_reduced.csv")

In [ ]:
def extract_binary_tags(df, schema_csv_path, new_name):
    """Identifies which binary columns are 1 and turns them into a text string."""
    all_cols = pl.read_csv(schema_csv_path, n_rows=0).columns
    tag_cols = [c for c in all_cols if c != 'BGGId' and c in df.columns]

    return df.select([
        pl.col("BGGId"),
        pl.concat_str(
            [pl.when(pl.col(c) == 1).then(pl.lit(c)).otherwise(pl.lit("")) for c in tag_cols],
            separator=", "
        ).str.replace_all(", , ", "").str.strip_chars(", ").alias(new_name)
    ])

In [ ]:
mech_tags = extract_binary_tags(items_df, f"{raw_data_path}/mechanics.csv", "mechanics_tags")
theme_tags = extract_binary_tags(items_df, f"{raw_data_path}/themes.csv", "themes_tags")
subcat_tags = extract_binary_tags(subcats, f"{raw_data_path}/subcategories.csv", "subcat_tags")
designer_tags = extract_binary_tags(designers, f"{raw_data_path}/designers_reduced.csv", "designer_tags")

# Joining all tags back to the main dataframe

In [ ]:
full_meta = items_df.join(mech_tags, on="BGGId", how="left") \
                    .join(theme_tags, on="BGGId", how="left") \
                    .join(subcat_tags, on="BGGId", how="left") \
                    .join(designer_tags, on="BGGId", how="left")

full_meta = full_meta.with_columns(
    pl.concat_str([
        pl.lit("Game: "), pl.col("Name"),
        pl.lit(". Published: "), pl.col("YearPublished").cast(pl.Utf8),
        pl.lit(". Designed by: "), pl.col("designer_tags").fill_null("Unknown Designer"),
        pl.lit(". Description: "), pl.col("Description"),
        pl.lit(". Mechanics: "), pl.col("mechanics_tags").fill_null("None"),
        pl.lit(". Themes: "), pl.col("themes_tags").fill_null("None"),
        pl.lit(". Subcategories: "), pl.col("subcat_tags").fill_null("None"),
        pl.lit(". Complexity (1-5): "), pl.col("GameWeight").cast(pl.Utf8),
        pl.lit(". Best with "), pl.col("BestPlayers").cast(pl.Utf8), pl.lit(" players."),
        pl.lit(". Expansions available: "), pl.col("NumExpansions").cast(pl.Utf8)
    ]).alias("v2_rich_profile")
)

print(f"Synthesized V2 profiles for {full_meta.height} games.")
print(f"V2 Sample: {full_meta['v2_rich_profile'][0][:350]}...")

Synthesized V2 profiles for 21925 games.
V2 Sample: Game: Die Macher. Published: 1986. Designed by: Karl-Heinz Schmiel. Description: die macher game seven sequential political race different region germany player charge national political party manage limited resource help party victory win party victory point regional election different way score victory point regional election supply eighty victor...


In [ ]:
full_meta.shape

(21925, 428)

In [ ]:
model = SentenceTransformer('all-mpnet-base-v2')

print(f"Generating V2 embeddings on {device}...")
v2_texts = full_meta["v2_rich_profile"].to_list()


v2_embeddings = model.encode(v2_texts, show_progress_bar=True, batch_size=64)

# Creating and Saving the V2 Embedding Parquet
embedding_v2_df = pl.DataFrame({
    "item_id": full_meta["item_id"],
    "BGGId": full_meta["BGGId"],
    "embedding_llm": v2_embeddings.tolist()
})

embedding_v2_df.write_parquet(f"{output_path}/item_embeddings_v2_768.parquet")
print(f"V2 'Total Context' Embeddings saved successfully!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Generating V2 embeddings on cuda...


Batches:   0%|          | 0/343 [00:00<?, ?it/s]

V2 'Total Context' Embeddings saved successfully!
